
# Ejemplo práctico: Clasificador Naive Bayes para detección de Spam

===============================================================

Este ejemplo demuestra el uso del clasificador Naive Bayes para clasificar
correos electrónicos como spam o no spam.

Conceptos aplicados:
- Entrenamiento y predicción de modelos de machine learning
- Teorema de Bayes para clasificación probabilística
- Procesamiento de texto (vectorización)

Requisitos:
    pip install pandas scikit-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os

In [ ]:
def cargar_datos(filepath):
    """
    Carga el dataset de correos electrónicos.

    Args:
        filepath: Ruta al archivo CSV (puede ser comprimido con .zip)

    Returns:
        DataFrame de pandas con los datos
    """
    print(f"Cargando datos desde: {filepath}")

    if filepath.endswith('.zip'):
        df = pd.read_csv(filepath, compression='zip')
    else:
        df = pd.read_csv(filepath)

    print(f"Total de correos cargados: {len(df)}")
    print(f"Columnas: {list(df.columns[:5])} ... {list(df.columns[-5:])}")

    return df


In [ ]:
def explorar_datos(df):
    """
    Muestra información básica sobre el dataset.
    """
    print("\n" + "=" * 60)
    print("EXPLORACIÓN DE DATOS")
    print("=" * 60)

    # Distribución de clases
    prediction_counts = df['Prediction'].value_counts()
    print(f"\nDistribución de clases (Prediction):")
    if 0 in prediction_counts:
        print(f"  No Spam (0): {prediction_counts[0]} ({prediction_counts[0]/len(df)*100:.1f}%)")
    if 1 in prediction_counts:
        print(f"  Spam (1): {prediction_counts[1]} ({prediction_counts[1]/len(df)*100:.1f}%)")

    # Muestra de datos (frecuencias de palabras)
    print("\n" + "-" * 40)
    print("Muestra de las primeras 5 filas (columnas seleccionadas):")
    print("-" * 40)
    cols_to_show = list(df.columns[1:6]) + ['Prediction']
    print(df[cols_to_show].head())


In [ ]:

def entrenar_modelo(X_train_vec, y_train):
    """
    Entrena el clasificador Naive Bayes Multinomial.

    El modelo calcula:
    - P(spam): Probabilidad a priori de que un correo sea spam
    - P(palabra|spam): Probabilidad condicional de cada palabra dado spam

    Args:
        X_train_vec: Matriz de características de entrenamiento
        y_train: Etiquetas de entrenamiento

    Returns:
        Modelo entrenado
    """
    print("\n" + "=" * 60)
    print("ENTRENAMIENTO DEL MODELO NAIVE BAYES")
    print("=" * 60)

    clf = MultinomialNB()
    clf.fit(X_train_vec, y_train)

    print("Modelo entrenado exitosamente!")

    return clf


In [ ]:

def evaluar_modelo(clf, X_test_vec, y_test):
    """
    Evalúa el rendimiento del modelo en el conjunto de prueba.
    """
    print("\n" + "=" * 60)
    print("EVALUACIÓN DEL MODELO")
    print("=" * 60)

    # Predicciones
    y_pred = clf.predict(X_test_vec)

    # Precisión
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nPrecisión global: {accuracy:.4f} ({accuracy*100:.2f}%)")

    # Reporte detallado
    print("\nReporte de clasificación:")
    print(classification_report(y_test, y_pred,
                               target_names=['No Spam', 'Spam'],
                               digits=4))

    # Matriz de confusión
    print("\nMatriz de confusión:")
    cm = confusion_matrix(y_test, y_pred)
    print(f"                Predicho")
    print(f"              No Spam  Spam")
    print(f"Real No Spam    {cm[0,0]:4d}   {cm[0,1]:4d}")
    print(f"     Spam       {cm[1,0]:4d}   {cm[1,1]:4d}")

    return y_pred

In [ ]:

def predecir_nuevos_correos(clf, vectorizer, correos):
    """
    Realiza predicciones sobre nuevos correos electrónicos.
    """
    print("\n" + "=" * 60)
    print("PREDICCIÓN DE NUEVOS CORREOS")
    print("=" * 60)

    # Vectorizar los nuevos correos
    correos_vec = vectorizer.transform(correos)

    # Predecir clase
    predicciones = clf.predict(correos_vec)

    # Predecir probabilidades
    probabilidades = clf.predict_proba(correos_vec)

    for i, correo in enumerate(correos):
        clase = "Spam" if predicciones[i] == 1 else "No Spam"
        prob_no_spam = probabilidades[i][0]
        prob_spam = probabilidades[i][1]

        print(f"\nCorreo {i+1}:")
        print(f"  Texto: {correo[:80]}...")
        print(f"  Predicción: {clase}")
        print(f"  Probabilidades:")
        print(f"    - No Spam: {prob_no_spam:.4f} ({prob_no_spam*100:.2f}%)")
        print(f"    - Spam: {prob_spam:.4f} ({prob_spam*100:.2f}%)")


In [ ]:

def mostrar_palabras_mas_relevantes(clf, vectorizer, n=10):
    """
    Muestra las palabras más indicativas de spam y no spam.
    """
    print("\n" + "=" * 60)
    print("PALABRAS MÁS RELEVANTES PARA LA CLASIFICACIÓN")
    print("=" * 60)

    # Obtener nombres de características
    nombres = vectorizer.get_feature_names_out()

    # Log probabilidades de cada característica dada cada clase
    log_prob = clf.feature_log_prob_

    # Diferencia entre probabilidades (spam - no spam)
    # Palabras con mayor diferencia son más indicativas de spam
    diferencia = log_prob[1] - log_prob[0]

    # Índices de las palabras más indicativas de spam
    indices_spam = diferencia.argsort()[-n:][::-1]
    # Índices de las palabras más indicativas de no spam
    indices_no_spam = diferencia.argsort()[:n]

    print(f"\nTop {n} palabras más indicativas de SPAM:")
    for i, idx in enumerate(indices_spam, 1):
        print(f"  {i:2d}. {nombres[idx]:15s} (peso: {diferencia[idx]:.4f})")

    print(f"\nTop {n} palabras más indicativas de NO SPAM:")
    for i, idx in enumerate(indices_no_spam, 1):
        print(f"  {i:2d}. {nombres[idx]:15s} (peso: {diferencia[idx]:.4f})")



In [ ]:
"""
Flujo principal del ejemplo de Naive Bayes.
"""
print("=" * 60)
print("CLASIFICADOR NAIVE BAYES - DETECCIÓN DE SPAM (Kaggle Dataset)")
print("=" * 60)

# Ruta al dataset
data_path = 'emails.csv.zip'


In [ ]:

# 1. CARGA DE DATOS
df = cargar_datos(data_path)
explorar_datos(df)

In [ ]:
# 2. SEPARAR CARACTERÍSTICAS Y ETIQUETAS
# Eliminamos 'Email No.' según recomendación y 'Prediction' que es nuestra etiqueta
X = df.drop(columns=['Email No.', 'Prediction'])
y = df['Prediction']


In [ ]:


# 3. DIVIDIR EN ENTRENAMIENTO Y PRUEBA
print("\n" + "=" * 60)
print("DIVISIÓN DE DATOS")
print("=" * 60)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% para prueba
    random_state=42,     # Para reproducibilidad
    stratify=y          # Mantiene proporción de clases
)
print(f"Datos de entrenamiento: {len(X_train)} correos")
print(f"Datos de prueba: {len(X_test)} correos")

In [ ]:
# 4. PREPROCESAMIENTO (Simulado para compatibilidad)
# El dataset ya está vectorizado, pero crearemos un vectorizer cargado con el vocabulario
# para que las funciones de predicción sigan funcionando.
X_train_vec = X_train
X_test_vec = X_test

# Crear un vectorizer manual con el vocabulario del dataset
vectorizer = CountVectorizer(vocabulary=list(X.columns))
# Marcamos como fit para que se pueda usar transform
vectorizer.vocabulary_ = {word: i for i, word in enumerate(X.columns)}
vectorizer.fixed_vocabulary_ = True

print("\n" + "=" * 60)
print("PREPROCESAMIENTO: Dataset ya vectorizado")
print("=" * 60)
print(f"Dimensiones de la matriz: {X_train_vec.shape}")


In [ ]:

# 5. ENTRENAMIENTO
clf = entrenar_modelo(X_train_vec, y_train)

In [ ]:
from math import exp

print(f"\nProbabilidades a priori aprendidas:")
print(f"  P(No Spam) = {exp(clf.class_log_prior_[0]):.4f}")
print(f"  P(Spam) = {exp(clf.class_log_prior_[1]):.4f}")

In [ ]:
# 6. EVALUACIÓN
evaluar_modelo(clf, X_test_vec, y_test)

In [ ]:
# 7. PALABRAS RELEVANTES
mostrar_palabras_mas_relevantes(clf, vectorizer)

In [ ]:
# 8. PREDICCIONES DE EJEMPLO
correos_ejemplo = [
    "Hello, can we schedule a meeting for tomorrow? Thanks, John",
    "Congratulations! You've won $1,000,000! Claim your prize now!",
    "URGENT: Your account needs verification. Click here immediately!",
    "Meeting notes from today's discussion about the project"
]
predecir_nuevos_correos(clf, vectorizer, correos_ejemplo)